# Exercise 2.2.2 — Data Types and Subsetting

This notebook continues with `datania_households_raw.csv` from 2.2.1. Using **only the tools from Lesson 2.2**, you fix data types and save a *typed checkpoint* that the cleaning step (2.2.3) will pick up.

You will practice:
- Telling a **DataFrame** from a **Series** and checking dtypes
- Standardising text and fixing categories with `.str` methods and `.replace()`
- Converting messy text to numbers with `.str.replace()` + `pd.to_numeric()`
- Converting text to dates with `pd.to_datetime()` and the `.dt` accessor
- Subsetting (boolean indexing, `isin()`, `str.contains()`) to *inspect* problems
- Saving a typed checkpoint to `10_cleaned/`

> **Pipeline:** reads `0_raw/` and writes a typed file to `10_cleaned/`. Exercise 2.2.3 reads that file.

### Path Setup (run first)

In [ ]:
import os
import numpy as np
import pandas as pd

DATA_RAW_DIR = '../../data/0_raw'
FILE_NAME = 'datania_households_raw.csv'
raw_path = os.path.join(DATA_RAW_DIR, FILE_NAME)

# Identifiers and codes must stay as text (preserve leading zeros)
df = pd.read_csv(raw_path, dtype={'hh_id': str, 'region_code': str})

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Loaded:', df.shape)
df.head()

---

## Task 1 — DataFrame vs Series

Selecting a single column returns a **Series** — a one-dimensional object with its own dtype and methods. Type checks (`.dtype`, `.str`, `.dt`, `.value_counts()`) operate on Series, not the whole DataFrame.

In [ ]:
print(type(df))
print(type(df['income_dkw']))

In [ ]:
# Print the dtype pandas assigned to each column
df.  # your code here

**Questions:**

- What dtype did pandas assign to `income_dkw`? Is it numeric? Why?
- What dtype is `survey_date`? What must you do before extracting the month from it?
- `hh_id` and `region_code` are text. What would have happened to `region_code` without the `dtype=` argument in `read_csv`?

---

## Task 2 — Standardise text and fix categories

Two replace methods look similar but differ:

- **`.replace({'old': 'new'})`** matches and replaces **whole values** (exact match)
- **`.str.replace('old', 'new')`** replaces a **substring** inside each string

Inspect the two categorical columns, then fix them with whole-value `.replace()`.

In [ ]:
print(df['urban_rural'].value_counts(dropna=False))
print()
print(df['region_code'].value_counts(dropna=False))

In [ ]:
# Fix the typo 'Urbn' -> 'Urban'
df['urban_rural'] = df['urban_rural'].  # your code here — .replace('Urbn', 'Urban')

# region_code: '99' is a coded-missing value -> NaN; '1' should be the 2-digit '01'.
# Why NOT .str.replace('1','01')? It would turn '01'->'001', '21'->'021'. Use whole-value .replace.
df['region_code'] = df['region_code'].  # your code here — .replace({'99': np.nan, '1': '01'})

print(df['urban_rural'].value_counts(dropna=False))
print()
print(df['region_code'].value_counts(dropna=False))

**Questions:**

- Why would `.str.replace('1', '01')` corrupt codes like `'01'`, `'16'`, or `'21'`?
- After the fix, how many `NaN` values does `region_code` have, and where did each come from?

---

## Task 3 — Convert text to numbers

`income_dkw` holds household income but arrives as text: spaces, commas, the `Ar` prefix, and labels like `unknown`. Remove the formatting with `.str.replace()`, replace the text labels (`unknown`, `NA`) with `NaN` using `.replace()`, then convert with `pd.to_numeric()`. Because every non-number is now `NaN`, use the **strict default** `errors='raise'` — if a label was missed the conversion fails loudly instead of hiding it.

In [ ]:
# Inspect the distinct raw values first
df['income_dkw'].unique()

In [ ]:
# Remove formatting, then turn the text labels into NaN so the column is fully numeric
df['income_dkw'] = (
    df['income_dkw']
    .astype('string')
    .str.replace(' ', '', regex=False)
    .str.replace('Ar', '', regex=False)
    # your code here — also remove commas (.str.replace),
    #                  then replace the labels 'unknown' and 'NA' with np.nan (.replace)
)
# Every label is now NaN, so the strict default conversion can be used
df['income_dkw'] = pd.to_numeric( # your code here — the column, errors='raise' )

df['income_dkw'].describe()

**Questions:**

- Which values end up as `NaN` after conversion, and what was each raw value?
- Why replace the text labels (`unknown`, `NA`) with `NaN` *before* converting, and why is the strict default `errors='raise'` a good choice once you have?
- `-5000` and `999999` survived as real numbers. Should they be left as-is here? (They are handled as *coded missing values* in 2.2.3.)

---

## Task 4 — Convert text to dates

Dates stored as text block any date analysis. A few values use inconsistent formats. Fix those known values with a whole-value `.replace()`, then parse everything with `pd.to_datetime()`. Once every malformed value is fixed or set to `NaN`, use the strict default `errors='raise'`. Use the `.dt` accessor to extract parts.

In [ ]:
df['survey_date'].unique()

In [ ]:
# Known bad values -> corrected ISO strings; 'not recorded' -> NaN
date_fixes = {
    'not recorded': np.nan,
    '03/15/2025':   '2025-03-15',   # MM/DD/YYYY
    '2025/01/18':   '2025-01-18',   # slash separators
    '2025-13-01':   '2025-01-13',   # day/month inverted
}
df['survey_date'] = df['survey_date'].replace(date_fixes)

# Every malformed value is fixed or NaN, so the strict default conversion can be used
df['survey_date'] = pd.to_datetime( # your code here — errors='raise' )

df['survey_date'].dtype

In [ ]:
# Extract the month with the .dt accessor (a demonstration — not saved to the checkpoint)
df['survey_date'].dt.  # your code here — month


**Questions:**

- Which raw date values needed fixing? What was wrong with each?
- What is `NaT`, and how does it differ from `NaN`?
- Try `df['survey_date'].dt.day_name()`. What other `.dt` properties are useful?

---

## Task 5 — Subsetting to inspect problems

Filtering lets you *inspect* specific rows without changing the data (in 2.3 you filter to *remove* rows). Use boolean indexing, combine conditions with `&` `|` `~` (wrap each in parentheses), and search text with `isin()` and `str.contains()`.

In [ ]:
# Households with negative income (a likely data error)
df[df['income_dkw'] < 0][['hh_id', 'income_dkw']]

In [ ]:
# Combine conditions: rural households with high income
df[(df['urban_rural'] == 'Rural') & (df['income_dkw'] > 50000)][['hh_id', 'urban_rural', 'income_dkw']]

In [ ]:
# Filter by a set of districts with isin()
target_districts = ['Polaris District', 'North Delta']
df[df['district'].isin( # your code here )][['hh_id', 'district']]

In [ ]:
# Case-insensitive text search, safe with missing values
df[df['district'].str.contains( # your code here — 'delta', case=False, na=False )][['hh_id', 'district']]

**Questions:**

- How many households have negative income? Which household(s)?
- Why must each condition be wrapped in parentheses when combining with `&`?
- What does `na=False` do in `str.contains()`? What happens without it?

---

## Task 6 — Save a typed checkpoint to `10_cleaned/`

Types are now fixed. Save a **typed checkpoint** so the cleaning step (2.2.3) can start from a stable baseline instead of re-reading raw. Keep the core columns with their corrected types — derived columns (like a `survey_month`) belong to the feature step (2.2.4), so we don't persist them here.

> **Never** modify files in `0_raw/`. Write to `10_cleaned/`.

In [ ]:
COLS_OUT = [
    'hh_id', 'region_code', 'province_name', 'district', 'urban_rural',
    'hh_size', 'income_dkw', 'survey_date', 'pop_density', 'education_code', 'age',
]
df_typed = df[COLS_OUT].copy()
print('Typed checkpoint:', df_typed.shape)
df_typed.head()

In [ ]:
DATA_CLEAN_DIR = '../../data/10_cleaned'
os.makedirs(DATA_CLEAN_DIR, exist_ok=True)
out_path = os.path.join(DATA_CLEAN_DIR, 'datania_households_clean.csv')

df_typed.to_csv( # your code here — index=False )
print('Saved:', out_path)

In [ ]:
# Reload to confirm it round-trips
check = pd.read_csv(out_path, dtype={'hh_id': str, 'region_code': str})
print('Reloaded:', check.shape)
print(check.dtypes)
check.head()

**Questions:**

- After reloading, what dtype does `survey_date` have? What does that tell you about CSV?
- Why `index=False`?
- This file is a *typed checkpoint*, not the final cleaned data. What still has to happen in 2.2.3?